# Exploração dos dados em Parquet — SAMU USB/USA

Este caderno carrega todos os arquivos `.parquet` da pasta `dados/` em
DataFrames do pandas e faz uma inspeção rápida de cada um:

- número de linhas e colunas
- quantas colunas são numéricas vs. categóricas/texto
- proporção de nulos por coluna
- cardinalidade (número de valores únicos)

Datasets encontrados em `dados/`:

| arquivo | conteúdo |
|---|---|
| `centroides.parquet` | centroide de cada município (lon/lat) |
| `frota.parquet` | uma linha por unidade móvel do SAMU (USB/USA), com atributos do CNES e do município |
| `municipal.parquet` | dados agregados por município (frota, cobertura, distâncias) |
| `municipios_ibge.parquet` | população, área e densidade por município (Censo 2022) |
| `sedes.parquet` | sede (centro urbano) de cada município |


In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

DADOS = Path("dados")


## 1. Carregar todos os parquet em um dicionário de DataFrames

In [2]:
arquivos = sorted(DADOS.glob("*.parquet"))
dfs = {p.stem: pd.read_parquet(p) for p in arquivos}

for nome, df in dfs.items():
    print(f"{nome:25s} {df.shape[0]:>6d} linhas x {df.shape[1]:>3d} colunas")


centroides                  5570 linhas x   3 colunas
frota                       7123 linhas x  54 colunas
municipal                   5570 linhas x  31 colunas
municipios_ibge             5570 linhas x   9 colunas
sedes                       5571 linhas x   9 colunas


## 2. Função auxiliar de resumo por coluna

In [3]:
def classifica_tipo(dtype) -> str:
    """Classifica um dtype do pandas em booleana / numérica / categórica-texto."""
    if pd.api.types.is_bool_dtype(dtype):
        return "booleana"
    if pd.api.types.is_numeric_dtype(dtype):
        return "numérica"
    return "categórica/texto"


def resumo_colunas(df: pd.DataFrame) -> pd.DataFrame:
    """Uma linha por coluna: dtype, tipo, nulos e cardinalidade."""
    linhas = []
    for col in df.columns:
        s = df[col]
        linhas.append({
            "coluna": col,
            "dtype": str(s.dtype),
            "tipo": classifica_tipo(s.dtype),
            "n_nulos": int(s.isna().sum()),
            "pct_nulos": round(100 * s.isna().mean(), 2),
            "n_unicos": int(s.nunique(dropna=True)),
        })
    return pd.DataFrame(linhas)


def resumo_dataset(nome: str, df: pd.DataFrame) -> None:
    """Imprime um resumo geral e mostra a tabela coluna a coluna."""
    resumo = resumo_colunas(df)
    contagem_tipos = resumo["tipo"].value_counts()

    print(f"=== {nome} ===")
    print(f"linhas: {df.shape[0]}  |  colunas: {df.shape[1]}")
    print(f"colunas por tipo: {contagem_tipos.to_dict()}")
    print(f"nulos no dataset: {round(100 * df.isna().to_numpy().mean(), 2)}% das células")
    display(resumo.sort_values("pct_nulos", ascending=False).reset_index(drop=True))


## 3. Explorar cada dataset

### `centroides.parquet`

Centroide de cada município (lon/lat) — chave para juntar com os demais por `cod_mun`.

In [4]:
df = dfs["centroides"]
resumo_dataset("centroides", df)


=== centroides ===
linhas: 5570  |  colunas: 3
colunas por tipo: {'numérica': 3}
nulos no dataset: 0.0% das células


,coluna,dtype,tipo,n_nulos,pct_nulos,n_unicos
0,cod_mun,int64,numérica,0,0.0,5570
1,lon_mun,float64,numérica,0,0.0,5570
2,lat_mun,float64,numérica,0,0.0,5570


In [5]:
df.head()

,cod_mun,lon_mun,lat_mun
0,1100015,-62.274992,-12.472449
1,1100023,-62.956006,-9.952527
2,1100031,-60.641062,-13.474839
3,1100049,-61.325362,-11.301189
4,1100056,-61.259708,-13.202918


### `frota.parquet`

Uma linha por unidade móvel (USB/USA) ativa, com dados do CNES cruzados com o município.

In [6]:
df = dfs["frota"]
resumo_dataset("frota", df)


=== frota ===
linhas: 7123  |  colunas: 54
colunas por tipo: {'categórica/texto': 38, 'numérica': 15, 'booleana': 1}
nulos no dataset: 21.76% das células


,coluna,dtype,tipo,n_nulos,pct_nulos,n_unicos
0,natureza_organizacao_entidade,object,categórica/texto,7123,100.00,0
1,descricao_nivel_hierarquia,object,categórica/texto,7123,100.00,0
2,estabelecimento_possui_centro_cirurgico,object,categórica/texto,7123,100.00,0
3,estabelecimento_possui_centro_obstetrico,object,categórica/texto,7123,100.00,0
4,estabelecimento_possui_centro_neonatal,object,categórica/texto,7123,100.00,0
5,estabelecimento_possui_atendimento_hospitalar,object,categórica/texto,7123,100.00,0
6,codigo_natureza_organizacao_unidade,object,categórica/texto,7123,100.00,0
7,codigo_nivel_hierarquia_unidade,object,categórica/texto,7123,100.00,0
8,numero_cnpj,str,categórica/texto,6410,89.99,607
9,codigo_motivo_desabilitacao_estabelecimento,str,categórica/texto,5991,84.11,10


In [7]:
df.head()

,codigo_cnes,numero_cnpj_entidade,nome_razao_social,nome_fantasia,natureza_organizacao_entidade,tipo_gestao,descricao_nivel_hierarquia,descricao_esfera_administrativa,codigo_tipo_unidade,codigo_cep_estabelecimento,endereco_estabelecimento,numero_estabelecimento,bairro_estabelecimento,numero_telefone_estabelecimento,latitude_estabelecimento_decimo_grau,longitude_estabelecimento_decimo_grau,endereco_email_estabelecimento,numero_cnpj,codigo_identificador_turno_atendimento,descricao_turno_atendimento,estabelecimento_faz_atendimento_ambulatorial_sus,codigo_estabelecimento_saude,codigo_uf,codigo_municipio,descricao_natureza_juridica_estabelecimento,codigo_motivo_desabilitacao_estabelecimento,estabelecimento_possui_centro_cirurgico,estabelecimento_possui_centro_obstetrico,estabelecimento_possui_centro_neonatal,estabelecimento_possui_atendimento_hospitalar,estabelecimento_possui_servico_apoio,estabelecimento_possui_atendimento_ambulatorial,codigo_atividade_ensino_unidade,codigo_natureza_organizacao_unidade,codigo_nivel_hierarquia_unidade,codigo_esfera_administrativa_unidade,data_atualizacao,ativo,cls_nome,CLASS_SR,cls_servico,modalidade,fonte_class,setor,cod_mun6,cod_mun,municipio,populacao,area_km2,densidade,uf_sigla,regiao,lat,lon
0,7627750,22855142000173,PREFEITURA MUNICIPAL DE MACHADINHO DO OESTE,SAMU 02 MACHADINHO,None,M,None,MUNICIPAL,42,76868000,COSTA E SILVA,2972,CENTRO,NaN,-9.423238,-62.002421,NaN,NaN,04,"ATENDIMENTO NOS TURNOS DA MANHA, TARDE E NOITE",NAO,1100137627750,11,110013,1244,NaN,None,None,None,None,1,0,04,None,None,M,2026-01-09,True,INDETERMINADO,002,USB,USB,servico_cnes,publico,110013,1100130,Machadinho D'Oeste - RO,30707.0,8509.270,3.61,RO,Norte,-9.423238,-62.002421
1,2808439,05903125000145,PREFEITURA DO MUNICIPIO DE PORTO VELHO,SAMU USA UNIDADE DE SUPORTE AVANCADO PORTO VELHO,None,M,None,MUNICIPAL,42,76820810,RUA VENEZUELA,2276,EMBRATEL,6932178400 992295694,-8.754363,-63.881665,192samupvh@gmail.com,NaN,06,ATENDIMENTO CONTINUO DE 24 HORAS/DIA (PLANTAO:...,SIM,1100202808439,11,110020,1244,NaN,None,None,None,None,1,0,04,None,None,M,2025-09-18,True,USA,003,USA,USA,servico_cnes,publico,110020,1100205,Porto Velho - RO,460434.0,34090.952,13.51,RO,Norte,-8.754363,-63.881665
2,8179069,63762058000192,PREFEITURA MUNICIPAL DE CACAULANDIA,SAMU CACAULANDIA 01,None,M,None,MUNICIPAL,42,76889000,RUA JOAO BOAVA,1400,SETOR 01,69992519612,-10.343025,-62.900605,samucacaulandia@gmail.com,NaN,06,ATENDIMENTO CONTINUO DE 24 HORAS/DIA (PLANTAO:...,NAO,1100608179069,11,110060,1244,NaN,None,None,None,None,0,0,04,None,None,M,2026-03-23,True,INDETERMINADO,NaN,NaN,INDETERMINADO,nao_classificada,publico,110060,1100601,Cacaulândia - RO,4150.0,1961.778,2.12,RO,Norte,-10.343025,-62.900605
3,8202125,04380507000179,MUNICIPIO DE OURO PRETO DO OESTE,SAMU OURO PRETO DO OESTE 02,None,M,None,MUNICIPAL,42,76920000,RUA JORGE TEIXEIRA,945,CENTRO RONDOMINAS,NaN,-10.512019,-62.000556,NaN,NaN,06,ATENDIMENTO CONTINUO DE 24 HORAS/DIA (PLANTAO:...,NAO,1100158202125,11,110015,1244,NaN,None,None,None,None,0,0,04,None,None,M,2026-04-09,True,INDETERMINADO,NaN,NaN,INDETERMINADO,nao_classificada,publico,110015,1100155,Ouro Preto do Oeste - RO,35044.0,1969.850,17.79,RO,Norte,-10.512019,-62.000556
4,8214441,63761902000160,UNI SOS EMERGENCIAS MEDICAS LTDA,UNI SOS EMERGENCIAS MEDICAS,None,M,None,MUNICIPAL,42,76860000,CASTELO BRANCO,593,SATELITE,16 32898599,-8.784781,-63.700104,COORDGERAL@UNI-SOS.COM.BR,10957463001929,06,ATENDIMENTO CONTINUO DE 24 HORAS/DIA (PLANTAO:...,NAO,1100808214441,11,110080,2062,NaN,None,None,None,None,1,0,04,None,None,M,2026-05-14,True,INDETERMINADO,NaN,NaN,INDETERMINADO,nao_classificada,privado,110080,1100809,Candeias do Jamari - RO,22310.0,6843.868,3.26,RO,Norte,-8.784781,-63.700104


### `municipal.parquet`

Agregado por município: contagem de unidades por modalidade, cobertura e distâncias sede-centroide.

In [8]:
df = dfs["municipal"]
resumo_dataset("municipal", df)


=== municipal ===
linhas: 5570  |  colunas: 31
colunas por tipo: {'numérica': 23, 'categórica/texto': 4, 'booleana': 4}
nulos no dataset: 0.0% das células


,coluna,dtype,tipo,n_nulos,pct_nulos,n_unicos
0,cod_mun,int64,numérica,0,0.0,5570
1,municipio,str,categórica/texto,0,0.0,5570
2,populacao,float64,numérica,0,0.0,5069
3,area_km2,float64,numérica,0,0.0,5556
4,densidade,float64,numérica,0,0.0,4056
5,uf_sigla,str,categórica/texto,0,0.0,27
6,uf_nome,str,categórica/texto,0,0.0,27
7,regiao,str,categórica/texto,0,0.0,5
8,cod_mun6,int64,numérica,0,0.0,5570
9,AEREA,int64,numérica,0,0.0,4


In [9]:
df.head()

,cod_mun,municipio,populacao,area_km2,densidade,uf_sigla,uf_nome,regiao,cod_mun6,AEREA,AQUATICA,INDETERMINADO,MOTOLANCIA,RESGATE/OUTROS,USA,USB,VIR,total,tem_usb,tem_usa,tem_qualquer,lon_cent,lat_cent,lat_sede,lon_sede,d_cent_sede_km,sede_corrigida,t_USB,t_USA,t_USB_cent,t_USA_cent
0,1100015,Alta Floresta D'Oeste - RO,21494.0,7067.127,3.04,RO,Rondônia,Norte,110001,0,0,0,0,0,0,0,0,0,False,False,False,-62.274992,-12.472449,-11.92830,-61.9953,67.713193,False,169.254280,169.278768,264.153460,264.174112
1,1100023,Ariquemes - RO,96833.0,4426.571,21.88,RO,Rondônia,Norte,110002,0,0,0,0,0,1,4,0,5,True,True,True,-62.956006,-9.952527,-9.90571,-63.0325,9.863978,False,1.965053,1.965053,13.569216,13.662864
2,1100031,Cabixi - RO,5351.0,1314.352,4.07,RO,Rondônia,Norte,110003,0,0,0,0,0,0,0,0,0,False,False,False,-60.641062,-13.474839,-13.49450,-60.5520,9.875274,False,275.583122,445.833415,289.887036,452.757795
3,1100049,Cacoal - RO,86887.0,3793.000,22.91,RO,Rondônia,Norte,110004,0,0,0,0,0,0,0,0,0,False,False,False,-61.325362,-11.301189,-11.43430,-61.4562,20.555166,False,115.183200,115.218018,116.120448,116.154231
4,1100056,Cerejeiras - RO,15890.0,2783.300,5.71,RO,Rondônia,Norte,110005,0,0,0,0,0,0,0,0,0,False,False,False,-61.259708,-13.202918,-13.18700,-60.8168,47.981531,False,323.310692,414.469883,392.047267,392.078663


### `municipios_ibge.parquet`

População, área e densidade por município (Censo 2022, IBGE).

In [10]:
df = dfs["municipios_ibge"]
resumo_dataset("municipios_ibge", df)


=== municipios_ibge ===
linhas: 5570  |  colunas: 9
colunas por tipo: {'numérica': 5, 'categórica/texto': 4}
nulos no dataset: 0.0% das células


,coluna,dtype,tipo,n_nulos,pct_nulos,n_unicos
0,cod_mun,int64,numérica,0,0.0,5570
1,municipio,str,categórica/texto,0,0.0,5570
2,populacao,float64,numérica,0,0.0,5069
3,area_km2,float64,numérica,0,0.0,5556
4,densidade,float64,numérica,0,0.0,4056
5,uf_sigla,str,categórica/texto,0,0.0,27
6,uf_nome,str,categórica/texto,0,0.0,27
7,regiao,str,categórica/texto,0,0.0,5
8,cod_mun6,int64,numérica,0,0.0,5570


In [11]:
df.head()

,cod_mun,municipio,populacao,area_km2,densidade,uf_sigla,uf_nome,regiao,cod_mun6
0,1100015,Alta Floresta D'Oeste - RO,21494.0,7067.127,3.04,RO,Rondônia,Norte,110001
1,1100023,Ariquemes - RO,96833.0,4426.571,21.88,RO,Rondônia,Norte,110002
2,1100031,Cabixi - RO,5351.0,1314.352,4.07,RO,Rondônia,Norte,110003
3,1100049,Cacoal - RO,86887.0,3793.000,22.91,RO,Rondônia,Norte,110004
4,1100056,Cerejeiras - RO,15890.0,2783.300,5.71,RO,Rondônia,Norte,110005


### `sedes.parquet`

Sede (coordenadas do centro urbano) de cada município.

In [12]:
df = dfs["sedes"]
resumo_dataset("sedes", df)


=== sedes ===
linhas: 5571  |  colunas: 9
colunas por tipo: {'numérica': 7, 'categórica/texto': 2}
nulos no dataset: 0.0% das células


,coluna,dtype,tipo,n_nulos,pct_nulos,n_unicos
0,codigo_ibge,int64,numérica,0,0.0,5571
1,nome,str,categórica/texto,0,0.0,5300
2,latitude,float64,numérica,0,0.0,5527
3,longitude,float64,numérica,0,0.0,5477
4,capital,int64,numérica,0,0.0,2
5,codigo_uf,int64,numérica,0,0.0,27
6,siafi_id,int64,numérica,0,0.0,5571
7,ddd,int64,numérica,0,0.0,67
8,fuso_horario,str,categórica/texto,0,0.0,4


In [13]:
df.head()

,codigo_ibge,nome,latitude,longitude,capital,codigo_uf,siafi_id,ddd,fuso_horario
0,5200050,Abadia de Goiás,-16.75730,-49.4412,0,52,1050,62,America/Sao_Paulo
1,3100104,Abadia dos Dourados,-18.48310,-47.3916,0,31,4001,34,America/Sao_Paulo
2,5200100,Abadiânia,-16.19700,-48.7057,0,52,9201,62,America/Sao_Paulo
3,3100203,Abaeté,-19.15510,-45.4444,0,31,4003,37,America/Sao_Paulo
4,1500107,Abaetetuba,-1.72183,-48.8788,0,15,401,91,America/Sao_Paulo


## 4. Resumo consolidado entre todos os datasets

In [14]:
linhas = []
for nome, df in dfs.items():
    resumo = resumo_colunas(df)
    linhas.append({
        "dataset": nome,
        "n_linhas": df.shape[0],
        "n_colunas": df.shape[1],
        "n_numericas": (resumo["tipo"] == "numérica").sum(),
        "n_categoricas": (resumo["tipo"] == "categórica/texto").sum(),
        "n_booleanas": (resumo["tipo"] == "booleana").sum(),
        "pct_nulos_medio": round(resumo["pct_nulos"].mean(), 2),
        "colunas_com_algum_nulo": int((resumo["n_nulos"] > 0).sum()),
    })

pd.DataFrame(linhas).set_index("dataset")


,n_linhas,n_colunas,n_numericas,n_categoricas,n_booleanas,pct_nulos_medio,colunas_com_algum_nulo
dataset,,,,,,,
centroides,5570,3,3,0,0,0.00,0
frota,7123,54,15,38,1,21.76,20
municipal,5570,31,23,4,4,0.00,0
municipios_ibge,5570,9,5,4,0,0.00,0
sedes,5571,9,7,2,0,0.00,0
